# Task A — Run 9 ensemble OOF evaluation

This notebook evaluates the second-best Task A recipe before any submission is made. It combines the original labelled training rows with the released labelled validation rows, generates five-fold out-of-fold predictions for demojized TF-IDF/SVM and demojized MuRIL with two reinitialized layers, then reports the locked Run 9 ensemble score. It does not fit on all rows and does not produce a submission ZIP.

The blend is fixed at 57% SVM + 43% MuRIL with a 0.50 threshold, matching the measured Run 9 recipe. Expected runtime is roughly 3–5 hours on a Kaggle T4 ×2.

In [ ]:
import os, pathlib, shutil, subprocess, sys
WORK = "/kaggle/working/hastika"
if os.path.isdir(WORK + "/.git"):
    subprocess.run(["git", "-C", WORK, "pull", "--ff-only", "origin", "task-b"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "-b", "task-b", "--depth", "1", "https://github.com/robinpnalex/Hastika-ICON2026.git", WORK], check=True)
os.chdir(WORK)
os.environ["PYTHONPATH"] = os.path.join(WORK, "src")
sys.path.insert(0, os.path.join(WORK, "src"))
pathlib.Path("artifacts/logs").mkdir(parents=True, exist_ok=True)
from urllib.request import urlretrieve
raw_base = "https://raw.githubusercontent.com/robinpnalex/Hastika-ICON2026/task-b/"
for rel in ["data/raw/hastika_binary_validation.csv", "data/raw/hastika_binary_test.csv"]:
    p = pathlib.Path(rel)
    if not p.exists():
        p.parent.mkdir(parents=True, exist_ok=True)
        print("downloading missing released data:", rel)
        urlretrieve(raw_base + rel, p)
    assert p.exists() and p.stat().st_size > 0, f"missing required data: {rel}"
subprocess.run('pip install -q emoji ftfy sentencepiece protobuf "transformers>=4.45,<6"', shell=True, check=True)
import numpy as np, pandas as pd, torch
from sklearn.metrics import f1_score, accuracy_score
from hastika.common.preprocessing import dedupe_index
assert torch.cuda.is_available(), "enable a GPU"
print("gpu:", torch.cuda.get_device_name(0))

def run(cmd, log=None):
    print("$", " ".join(map(str, cmd)), flush=True)
    fh = open(log, "w") if log else None
    p = subprocess.Popen([str(x) for x in cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        sys.stdout.write(line)
        if fh: fh.write(line)
    p.wait()
    if fh: fh.close()
    if p.returncode: raise RuntimeError(f"exit {p.returncode}: {cmd}")


## 1. Stage the combined labelled corpus

In [ ]:
from pathlib import Path
train_path = Path("data/raw/binary_train.csv")
val_path = Path("data/raw/hastika_binary_validation.csv")
combined = pd.concat([pd.read_csv(train_path), pd.read_csv(val_path)], ignore_index=True)
combined = combined.drop_duplicates("id", keep="first").reset_index(drop=True)
assert len(combined) == 7252 and combined["id"].is_unique
combined_path = Path("data/derived/task_a_run9_oof_train_plus_validation.csv")
combined_path.parent.mkdir(parents=True, exist_ok=True)
combined.to_csv(combined_path, index=False)
CANON_TRAIN = Path("data/raw/binary_train.csv")
BACKUP_TRAIN = Path("data/raw/binary_train.csv.before_run9_oof")
assert not BACKUP_TRAIN.exists(), "stale backup exists; stop and inspect before rerunning"
shutil.copy2(CANON_TRAIN, BACKUP_TRAIN)
shutil.copy2(combined_path, CANON_TRAIN)
print("combined labelled rows:", len(combined))


## 2. Generate matched five-fold OOF probabilities

In [ ]:
SVM_TAG = "a_run9_oof_svm"
MURIL_TAG = "a_run9_oof_muril"
run([sys.executable, "-u", "-m", "hastika.models.baseline_svm", "--task", "a", "--tag", SVM_TAG, "--demojize"], log=f"artifacts/logs/{SVM_TAG}.log")
run([sys.executable, "-u", "-m", "hastika.models.muril", "--tag", MURIL_TAG, "--model", "google/muril-base-cased", "--folds", "5", "--seeds", "42", "--epochs", "6", "--bs", "8", "--grad-accum", "2", "--eval-bs", "32", "--select", "last", "--reinit-layers", "2"], log=f"artifacts/logs/{MURIL_TAG}.log")
print("SVM and two-layer MuRIL OOF runs completed")


## 3. Report the Run 9 ensemble performance

In [ ]:
keep = dedupe_index(combined["Comment"].tolist(), combined["Label"].tolist(), "task A")
fit_df = combined.iloc[keep].reset_index(drop=True)
y = (fit_df["Label"] == "Hate").astype(int).to_numpy()
svm = np.load(Path("artifacts/runs") / SVM_TAG / "oof_probs.npy")
muril = np.load(Path("artifacts/runs") / MURIL_TAG / "oof_probs.npy")
assert svm.shape == muril.shape == (len(y), 2)
p_svm, p_muril = svm[:, 1], muril[:, 1]
p_ensemble = 0.57 * p_svm + 0.43 * p_muril
pred_svm = (p_svm > 0.50).astype(int)
pred_muril = (p_muril > 0.50).astype(int)
pred_ensemble = (p_ensemble > 0.50).astype(int)
summary = pd.DataFrame([
    {"model": "demojized TF-IDF/SVM", "macro_f1": f1_score(y, pred_svm, average="macro"), "accuracy": accuracy_score(y, pred_svm)},
    {"model": "Run 9 two-layer MuRIL", "macro_f1": f1_score(y, pred_muril, average="macro"), "accuracy": accuracy_score(y, pred_muril)},
    {"model": "Run 9 locked ensemble", "macro_f1": f1_score(y, pred_ensemble, average="macro"), "accuracy": accuracy_score(y, pred_ensemble)},
])
print(summary.to_string(index=False))
OUT = Path("/kaggle/working/task_a_run9_oof_outputs")
OUT.mkdir(parents=True, exist_ok=True)
summary.to_csv(OUT / "summary.csv", index=False)
np.savez(OUT / "oof_probabilities.npz", y=y, svm=svm, muril=muril, ensemble=p_ensemble)
print("DOWNLOAD RESULTS FROM:", OUT)


## 4. Restore the canonical training file

In [ ]:
shutil.move(str(BACKUP_TRAIN), str(CANON_TRAIN))
print("restored", CANON_TRAIN)
